# Combined analysis of the counting experiment

## Motivation

Each collaboration measures the same physical quantity: the half-life $\mathcal{T}_{1/2}^{\beta\beta0\nu}$.
The individual measurements can be combined into a single, more powerful result.

## Method

Each collaboration $c$ observes $n_c$ events in the RoI, drawn from a Poisson distribution:

$$n_c \sim \mathrm{Poisson}(b_c + \mu \, s_c)$$

where:
- $b_c$ is the expected background (estimated from the blind analysis)
- $\mu = 1 / \mathcal{T}_{1/2}$ is the **common parameter** (decay rate)
- $s_c = \frac{10^3 \, \delta \, \epsilon_c \, E_c \, N_A \, \ln 2}{W}$ is the sensitivity factor of collaboration $c$
  ($\epsilon_c$ = signal efficiency, $E_c$ = exposure in kg$\cdot$y)

The combined likelihood is the product of the individual Poisson likelihoods:

$$\mathcal{L}(\mu) = \prod_c \mathrm{Poisson}(n_c \mid b_c + \mu \, s_c)$$

We scan $-2\log\mathcal{L}(\mu)$ to find the best estimate $\hat{\mu}$, confidence intervals,
and the p-value of the null hypothesis ($\mu = 0$).

## Setup

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import scipy.optimize as optimize
import scipy.constants as constants

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os, sys
from pathlib import Path

_env = os.environ.get('FANAL_ROOT') or os.environ.get('USCFANALDIR')
if _env and Path(_env, 'data').is_dir():
    rootpath = str(Path(_env).resolve())
else:
    rootpath = str(next(p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'data').is_dir() and (p / 'ana').is_dir()))
if rootpath not in sys.path:
    sys.path.insert(0, rootpath)

import core.pltext as pltext
pltext.style()

print('Fanal root:', rootpath)

## Load parameters from all collaborations

In [ ]:
import importlib

collab_names = ['new_alpha', 'new_beta', 'new_gamma', 'new_delta', 'new_epsilon']
short_names  = ['Alpha', 'Beta', 'Gamma', 'Delta', 'Epsilon']

# Load each collpars module
collpars = {}
for name in collab_names:
    mod = importlib.import_module('collpars_' + name)
    collpars[name] = mod

# Physical constants (same for all)
delta = 0.9            # 136Xe abundance
W     = 135.9          # g/mol
NA    = constants.Avogadro

# Extract per-collaboration quantities
n_obs = np.array([collpars[c].n_obs_RoI    for c in collab_names])  # observed counts
bkg   = np.array([collpars[c].n_Bi_RoI + collpars[c].n_Tl_RoI for c in collab_names])  # expected bkg
eff   = np.array([collpars[c].eff_bb_RoI   for c in collab_names])  # signal efficiency
expo  = np.array([collpars[c].exposure      for c in collab_names])  # exposure (kg*y)

# Sensitivity factor: expected signal events per unit mu = 1/T_half
sens = 1e3 * delta * eff * expo * NA * np.log(2.) / W

print(f'{"Collab":>10s} {"n_obs":>6s} {"bkg":>8s} {"eff":>8s} {"expo":>8s} {"sens":>12s}')
print('-' * 58)
for i, c in enumerate(short_names):
    print(f'{c:>10s} {n_obs[i]:6d} {bkg[i]:8.2f} {eff[i]:8.4f} {expo[i]:8.0f} {sens[i]:12.2e}')

## Combined Poisson likelihood

The negative log-likelihood for the combined measurement is:

$$-2 \log \mathcal{L}(\mu) = -2 \sum_c \left[ n_c \log(b_c + \mu s_c) - (b_c + \mu s_c) - \log(n_c!) \right]$$

Since $\log(n_c!)$ is constant, we drop it and work with:

$$-2 \Delta\log \mathcal{L}(\mu) = -2 \log \mathcal{L}(\mu) + 2 \log \mathcal{L}(\hat{\mu})$$

In [ ]:
def nll(mu, n_obs, bkg, sens):
    """Combined -2 log-likelihood (up to a constant)."""
    lam = bkg + mu * sens
    lam = np.maximum(lam, 1e-30)
    return np.sum(-2 * (n_obs * np.log(lam) - lam))


def nll_single(mu, n_c, b_c, s_c):
    """-2 log-likelihood for a single collaboration."""
    lam = max(b_c + mu * s_c, 1e-30)
    return -2 * (n_c * np.log(lam) - lam)

## Find the best-fit $\hat{\mu}$

In [ ]:
# Analytic MLE as starting point: sum(n_obs) = sum(bkg + mu*sens)
mu_start = max((np.sum(n_obs) - np.sum(bkg)) / np.sum(sens), 1e-30)

# Minimize the combined NLL using Brent's method with a bracket
result = optimize.minimize_scalar(lambda mu: nll(mu, n_obs, bkg, sens),
                                  bracket=(mu_start / 10, mu_start, mu_start * 10))
mu_hat = result.x
nll_hat = nll(mu_hat, n_obs, bkg, sens)

tau_hat = 1. / mu_hat
print(f'Best-fit mu  = {mu_hat:.4e}  (1/y)')
print(f'Best-fit T½  = {tau_hat:.3e}  y  ({tau_hat/1e25:.2f} x 10^25 y)')

## Profile likelihood scan

In [ ]:
# Scan mu and compute Delta(-2 log L)
mu_scan = np.linspace(0, 5 * mu_hat, 500)
dnll_scan = np.array([nll(mu, n_obs, bkg, sens) for mu in mu_scan])
dnll_scan = dnll_scan - nll_hat  # Delta NLL (positive, minimum = 0)

plt.figure(figsize=(8, 5))
plt.plot(mu_scan * 1e26, dnll_scan, 'b-', lw=2)
plt.axhline(1, color='orange', ls='--', label=r'$\Delta(-2\log\mathcal{L}) = 1$ (68% CL)')
plt.axhline(3.84, color='red', ls='--', label=r'$\Delta(-2\log\mathcal{L}) = 3.84$ (95% CL)')
plt.axvline(mu_hat * 1e26, color='gray', ls=':', label=rf'$\hat{{\mu}}$ = {mu_hat:.2e}')
plt.xlabel(r'$\mu$ ($\times 10^{-26}$ y$^{-1}$)')
plt.ylabel(r'$\Delta(-2\log\mathcal{L})$')
plt.title('Combined profile likelihood scan')
plt.ylim(0, 10)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

## Confidence intervals on $\mu$ and $\mathcal{T}_{1/2}$

In [ ]:
def find_ci(mu_scan, dnll_scan, mu_hat, level=1.0):
    """Find the CI on mu from the Delta NLL scan.
    level = 1.0 for 68% CL, 3.84 for 95% CL.
    Returns (mu_lo, mu_hi).
    """
    # Find all crossings of dnll_scan = level
    above = dnll_scan > level
    crossings = np.where(np.diff(above.astype(int)))[0]

    mu_lo = mu_scan[0]  # default: boundary
    mu_hi = mu_scan[-1]

    for idx in crossings:
        # Linear interpolation between idx and idx+1
        frac = (level - dnll_scan[idx]) / (dnll_scan[idx + 1] - dnll_scan[idx])
        mu_cross = mu_scan[idx] + frac * (mu_scan[idx + 1] - mu_scan[idx])
        if mu_cross < mu_hat:
            mu_lo = mu_cross
        else:
            mu_hi = mu_cross
            break  # take the first crossing above mu_hat

    return mu_lo, mu_hi

ci_68 = find_ci(mu_scan, dnll_scan, mu_hat, level=1.0)
ci_95 = find_ci(mu_scan, dnll_scan, mu_hat, level=3.84)

print('Combined results on mu (1/T½):')
print(f'  mu_hat = {mu_hat:.3e} 1/y')
print(f'  68% CL: ({ci_68[0]:.3e}, {ci_68[1]:.3e})')
print(f'  95% CL: ({ci_95[0]:.3e}, {ci_95[1]:.3e})')
print()
print('Combined results on T½:')
tau_hat = 1. / mu_hat
tau_68_hi = 1. / ci_68[0] if ci_68[0] > 0 else float('inf')
tau_68  = (1. / ci_68[1], tau_68_hi)
tau_95_hi = 1. / ci_95[0] if ci_95[0] > 0 else float('inf')
tau_95  = (1. / ci_95[1], tau_95_hi)
print(f'  T½     = {tau_hat:.3e} y  ({tau_hat/1e25:.2f} x 10^25 y)')
print(f'  68% CL: ({tau_68[0]/1e25:.2f}, {tau_68[1]/1e25:.2f}) x 10^25 y')
print(f'  95% CL: ({tau_95[0]/1e25:.2f}, {tau_95[1]/1e25:.2f}) x 10^25 y')

## p-value of the null hypothesis ($\mu = 0$)

The test statistic is:

$$q_0 = -2\log\frac{\mathcal{L}(\mu=0)}{\mathcal{L}(\hat{\mu})}$$

Under the null hypothesis, $q_0 \sim \chi^2(1)/2$, so $p_0 = 1 - \Phi(\sqrt{q_0})$.

In [ ]:
nll_0 = nll(0, n_obs, bkg, sens)
q0 = nll_0 - nll_hat
z0 = np.sqrt(max(q0, 0))
p0 = 1 - stats.norm.cdf(z0)

print(f'q0      = {q0:.2f}')
print(f'z0      = {z0:.2f} sigma')
print(f'p-value = {p0:.2e}')
print()
if z0 >= 5:
    print('=> Discovery! (>= 5 sigma)')
elif z0 >= 3:
    print('=> Evidence (>= 3 sigma)')
else:
    print('=> No significant excess (< 3 sigma)')

## Comparison: individual vs combined

Let us compare the individual results with the combined measurement.

In [ ]:
# Individual results
print(f'{"Collab":>10s} {"n_obs":>6s} {"bkg":>8s} {"n_sig":>8s} {"z0":>6s} {"T½ (1e25y)":>12s}')
print('-' * 56)

individual_tau = []
individual_z0  = []
for i, c in enumerate(short_names):
    cp = collpars[collab_names[i]]
    tau_c = cp.half_life_count / 1e25
    individual_tau.append(tau_c)
    individual_z0.append(cp.z0)
    print(f'{c:>10s} {n_obs[i]:6d} {bkg[i]:8.2f} {cp.n_signal_RoI:8.2f} {cp.z0:6.2f} {tau_c:12.2f}')

print('-' * 56)
print(f'{"COMBINED":>10s} {int(np.sum(n_obs)):6d} {np.sum(bkg):8.2f} '
      f'{np.sum(n_obs) - np.sum(bkg):8.2f} {z0:6.2f} {tau_hat/1e25:12.2f}')

In [ ]:
# Graphical comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: T½ comparison
ax = axes[0]
y_pos = np.arange(len(short_names) + 1)
labels = short_names + ['COMBINED']
tau_vals = individual_tau + [tau_hat / 1e25]

colors = ['steelblue'] * len(short_names) + ['red']
ax.barh(y_pos, tau_vals, color=colors, alpha=0.7, edgecolor='k', lw=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(labels)
ax.set_xlabel(r'$\mathcal{T}_{1/2}$ ($\times 10^{25}$ y)')
ax.set_title(r'Half-life: individual vs combined')
ax.grid(alpha=0.3, axis='x')

# Right: significance comparison
ax = axes[1]
z_vals = individual_z0 + [z0]
ax.barh(y_pos, z_vals, color=colors, alpha=0.7, edgecolor='k', lw=0.5)
ax.axvline(3, color='orange', ls='--', lw=1.5, label='3$\sigma$ (evidence)')
ax.axvline(5, color='red', ls='--', lw=1.5, label='5$\sigma$ (discovery)')
ax.set_yticks(y_pos)
ax.set_yticklabels(labels)
ax.set_xlabel(r'Significance ($\sigma$)')
ax.set_title('Significance: individual vs combined')
ax.legend(loc='lower right')
ax.grid(alpha=0.3, axis='x')

plt.tight_layout()

## Individual likelihood contributions

Visualise how each collaboration contributes to the combined likelihood.

In [ ]:
plt.figure(figsize=(9, 6))
colors_ind = ['royalblue', 'seagreen', 'darkorange', 'purple', 'brown']

for i, c in enumerate(short_names):
    dnll_i = np.array([nll_single(mu, n_obs[i], bkg[i], sens[i]) for mu in mu_scan])
    nll_hat_i = min(dnll_i)
    dnll_i = dnll_i - nll_hat_i
    plt.plot(mu_scan * 1e26, dnll_i, '-', color=colors_ind[i], lw=1.5,
             alpha=0.7, label=c)

plt.plot(mu_scan * 1e26, dnll_scan, 'k-', lw=2.5, label='Combined')
plt.axhline(1, color='orange', ls='--', alpha=0.5)
plt.axvline(mu_hat * 1e26, color='gray', ls=':')

plt.xlabel(r'$\mu$ ($\times 10^{-26}$ y$^{-1}$)')
plt.ylabel(r'$\Delta(-2\log\mathcal{L})$')
plt.title('Individual and combined likelihood profiles')
plt.ylim(0, 10)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

## Compatibility test

Test whether the individual measurements are consistent with each other.
Under the null hypothesis that all collaborations measure the same $\mu$,
the sum of individual $-2\log\mathcal{L}(\hat{\mu})$ minus the combined
$-2\log\mathcal{L}(\hat{\mu})$ should follow a $\chi^2(N-1)$ distribution.

In [ ]:
# Individual best fits
mu_hats_ind = []
nll_hats_ind = []
for i in range(len(short_names)):
    mu_start_i = max((n_obs[i] - bkg[i]) / sens[i], 1e-30)
    res_i = optimize.minimize_scalar(
        lambda mu, i=i: nll_single(mu, n_obs[i], bkg[i], sens[i]),
        bracket=(mu_start_i / 10, mu_start_i, mu_start_i * 10))
    mu_hats_ind.append(res_i.x)
    nll_hats_ind.append(nll_single(res_i.x, n_obs[i], bkg[i], sens[i]))

# Sum of individual best NLLs
sum_nll_ind = sum(nll_hats_ind)

# Compatibility chi2: combined NLL - sum of individual best NLLs
chi2_compat = nll_hat - sum_nll_ind
ndof = len(short_names) - 1
p_compat = stats.chi2.sf(chi2_compat, ndof)

print('Compatibility test:')
print(f'  chi2 = {chi2_compat:.2f},  ndof = {ndof},  p-value = {p_compat:.3f}')
if p_compat > 0.05:
    print('  => Results are compatible (p > 0.05)')
else:
    print('  => Results show tension (p < 0.05)')

print()
print('Individual best-fit T½:')
for i, c in enumerate(short_names):
    tau_i = 1. / mu_hats_ind[i] if mu_hats_ind[i] > 0 else float('inf')
    print(f'  {c:>10s}: T½ = {tau_i/1e25:.2f} x 10^25 y')
print(f'  {"COMBINED":>10s}: T½ = {tau_hat/1e25:.2f} x 10^25 y')

## Summary

| Quantity | Combined | Best individual |
|---|---|---|
| $\hat{\mu}$ | — | — |
| $\mathcal{T}_{1/2}$ | — | — |
| Significance ($z_0$) | — | — |
| p-value | — | — |

**Key observations:**
- The combined significance is higher than any individual measurement.
- The combined half-life represents a weighted average that naturally accounts
  for the different sensitivities (exposure, efficiency, background) of each collaboration.
- The compatibility test verifies that the individual results are consistent
  with a common signal.